<a href="https://colab.research.google.com/github/mirzafani808-source/prompt-engineering-structured-outputs/blob/main/prompt_engineering_structured_outputs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3: Prompt Engineering & Structured Outputs
**Use case:** Customer Support Ticket Triage. Raw ticket text goes in, validated JSON comes out.

What this notebook demonstrates:
1. Bad prompt vs improved prompt on the same input
2. Structured JSON output with a schema (Pydantic)
3. Few-shot examples for consistency
4. Context window management (trim + summarize history)
5. Safe parsing + validation + retry before the app uses the output

**API:** Groq (free tier, no credit card). Get a key at https://console.groq.com/keys
In Colab: click the key icon in the left sidebar, add a secret named `GROQ_API_KEY`, and enable notebook access.

In [ ]:
!pip -q install groq pydantic

## 1. Define the output schema
The schema is the contract. If the model breaks it, we catch it before it reaches our app.

In [ ]:
import os, json, re, time, getpass
from collections import Counter
from typing import Literal, List
from pydantic import BaseModel, Field, ValidationError
from groq import Groq

try:
    from google.colab import userdata
    API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    API_KEY = os.environ.get("GROQ_API_KEY") or getpass.getpass("Paste your Groq API key: ")

client = Groq(api_key=API_KEY)

# ---- Auto-pick a model that currently exists on Groq ----
available = [m.id for m in client.models.list().data]
print("Available models:", available)

SKIP = ("whisper", "tts", "guard", "compound", "orpheus", "embed", "safeguard")
PREFER = ("llama-3.3-70b", "llama-3.1-70b", "llama-4", "gpt-oss-120b", "qwen", "gpt-oss-20b", "llama-3.1-8b")

chat_models = [m for m in available if not any(s in m.lower() for s in SKIP)]
MODEL = None
for p in PREFER:
    for m in chat_models:
        if p in m:
            MODEL = m
            break
    if MODEL:
        break
MODEL = MODEL or chat_models[0]
print("Using model:", MODEL)

def call_llm(messages, json_mode=False, temperature=0.0, max_tokens=800):
    """Single place where we talk to the model."""
    kwargs = dict(model=MODEL, messages=messages, temperature=temperature, max_tokens=max_tokens)
    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}
    resp = client.chat.completions.create(**kwargs)
    return resp.choices[0].message.content

print(call_llm([{"role": "user", "content": "Say hi in 3 words."}]))

Available models: ['allam-2-7b', 'canopylabs/orpheus-arabic-saudi', 'openai/gpt-oss-120b', 'meta-llama/llama-prompt-guard-2-86m', 'whisper-large-v3', 'meta-llama/llama-prompt-guard-2-22m', 'canopylabs/orpheus-v1-english', 'openai/gpt-oss-safeguard-20b', 'whisper-large-v3-turbo', 'qwen/qwen3.8-27b', 'openai/gpt-oss-20b']
Using model: openai/gpt-oss-120b
Hello there, friend!


In [ ]:
class Ticket(BaseModel):
    category: Literal["billing", "technical", "account", "shipping", "other"]
    priority: Literal["low", "medium", "high", "urgent"]
    sentiment: Literal["positive", "neutral", "negative"]
    summary: str = Field(min_length=5, max_length=200)
    action_items: List[str] = Field(min_length=1, max_length=5)
    needs_human: bool

SAMPLE = """Hi, I was charged twice for my Pro subscription this month (Order #88213).
I have emailed twice already and nobody replied. This is really frustrating.
Please refund the extra charge today or I will dispute it with my bank."""

## 2. Bad prompt vs improved prompt (same input)
This is the comparison for the LinkedIn video.

In [ ]:
# ---------- BAD PROMPT ----------
bad_prompt = f"Analyze this support ticket: {SAMPLE}"
bad_output = call_llm([{"role": "user", "content": bad_prompt}], temperature=0.7, max_tokens=400)
print("=== BAD PROMPT OUTPUT ===")
print(bad_output)

=== BAD PROMPT OUTPUT ===
**Thank you for reaching out, and I’m very sorry to hear about the duplicate charge on your Pro subscription.** I understand how frustrating this can be, especially when you haven’t received a response to your earlier emails. Let’s get this sorted out as quickly as possible.

---

## 1. Verify the Charge
To investigate and process a refund for the extra charge, I’ll need a few details from you:

| Information Needed | Why It Helps |
|--------------------|--------------|
| **Order number** (you’ve already provided **#88213**) | Lets us locate the exact transaction in our system. |
| **Date of each charge** (e.g., “April 12 2024” and “April 13 2024”) | Confirms which of the two charges is the duplicate. |
| **Last four digits of the payment card** used for the subscription | Helps us match the payment method without exposing full card details. |
| **A screenshot of your billing statement** (optional, but useful) | Provides visual proof of the double charge and s

In [ ]:
# ---------- IMPROVED PROMPT ----------
SYSTEM_PROMPT = """You are a support-ticket triage engine. Read the ticket and return ONLY a JSON object.

Schema:
{
  "category": "billing" | "technical" | "account" | "shipping" | "other",
  "priority": "low" | "medium" | "high" | "urgent",
  "sentiment": "positive" | "neutral" | "negative",
  "summary": string (one sentence, max 200 chars),
  "action_items": [string] (1 to 5 short imperative steps),
  "needs_human": boolean
}

Rules:
- Output JSON only. No markdown, no code fences, no explanation.
- Use ONLY the allowed values above. Never invent new categories.
- "urgent" = money loss, security issue, or service fully down.
- needs_human = true if the customer is angry, threatens legal/bank action, or the issue involves money.
- Use only facts in the ticket. If unsure, pick "other" and priority "medium"."""

FEW_SHOT = [
    ("I can't log in since yesterday, reset email never arrives. Please help!",
     {"category": "account", "priority": "high", "sentiment": "negative",
      "summary": "Customer cannot log in and is not receiving the password reset email.",
      "action_items": ["Verify email address on file", "Check email delivery logs", "Send manual reset link"],
      "needs_human": False}),
    ("Just wanted to say the new dashboard is great, thanks team! One small thing: dark mode would be nice.",
     {"category": "other", "priority": "low", "sentiment": "positive",
      "summary": "Customer praises the new dashboard and requests a dark mode feature.",
      "action_items": ["Log dark mode as feature request", "Reply with thanks"],
      "needs_human": False}),
    ("My package was supposed to arrive last week. Tracking has said 'in transit' for 9 days. Where is it?",
     {"category": "shipping", "priority": "medium", "sentiment": "negative",
      "summary": "Package has been stuck in transit for 9 days past the expected delivery date.",
      "action_items": ["Contact courier for status", "Offer reshipment or refund if lost"],
      "needs_human": False}),
]

def build_messages(ticket_text, use_few_shot=True):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]
    if use_few_shot:
        for text, out in FEW_SHOT:
            msgs.append({"role": "user", "content": f"Ticket:\n{text}"})
            msgs.append({"role": "assistant", "content": json.dumps(out)})
    msgs.append({"role": "user", "content": f"Ticket:\n{ticket_text}"})
    return msgs

good_output = call_llm(build_messages(SAMPLE), json_mode=True, temperature=0.0)
print("=== IMPROVED PROMPT OUTPUT ===")
print(good_output)

=== IMPROVED PROMPT OUTPUT ===
{"category":"billing","priority":"urgent","sentiment":"negative","summary":"Customer was double‑charged for Pro subscription, demands refund and threatens bank dispute.","action_items":["Verify duplicate charge on account","Process refund for the extra charge","Confirm refund completion with customer","Escalate issue to billing manager","Document dispute threat"],"needs_human":true}


## 3. Safe parsing + validation + retry
Never trust raw model output. Steps: extract JSON, parse, validate against the schema,
and if that fails, send the error back to the model and retry.

In [ ]:
def extract_json(text: str) -> str:
    """Strip code fences / chatter around the JSON object."""
    text = text.strip()
    m = re.search(r"```(?:json)?\s*(.*?)```", text, re.S)
    if m:
        text = m.group(1).strip()
    start, end = text.find("{"), text.rfind("}")
    if start != -1 and end > start:
        text = text[start:end + 1]
    return text

def safe_parse(raw: str):
    """Returns (Ticket | None, error_message | None). Never raises."""
    try:
        return Ticket.model_validate_json(extract_json(raw)), None
    except ValidationError as e:
        return None, str(e)[:600]
    except Exception as e:
        return None, f"Unexpected error: {e}"

def triage_ticket(ticket_text: str, use_few_shot=True, temperature=0.0, max_retries=2):
    """Full pipeline: prompt -> parse -> validate -> retry with error feedback."""
    messages = build_messages(ticket_text, use_few_shot)
    last_err = None
    for attempt in range(max_retries + 1):
        raw = call_llm(messages, json_mode=True, temperature=temperature)
        parsed, err = safe_parse(raw)
        if parsed:
            return {"ok": True, "data": parsed, "attempts": attempt + 1}
        last_err = err
        messages += [
            {"role": "assistant", "content": raw},
            {"role": "user", "content": f"Your JSON was invalid: {err}\nReturn ONLY the corrected JSON object."},
        ]
    return {"ok": False, "error": last_err, "attempts": max_retries + 1}

result = triage_ticket(SAMPLE)
print(result["ok"], "| attempts:", result["attempts"])
if result["ok"]:
    print(result["data"].model_dump_json(indent=2))

# Prove the validator works on broken output:
for broken in ['Sure! Here you go: {"category": "billing", "priority": "super-urgent"}',
               '```json\n{"category": "billing"}\n```',
               'not json at all']:
    p, e = safe_parse(broken)
    print("\nINPUT:", broken[:60].replace("\n", " "), "\n -> parsed:", bool(p), "| error:", (e or "")[:90].replace("\n", " "))

True | attempts: 1
{
  "category": "billing",
  "priority": "urgent",
  "sentiment": "negative",
  "summary": "Customer was double‑charged for Pro subscription, demands refund and threatens bank dispute.",
  "action_items": [
    "Verify duplicate charge on account",
    "Process refund for the extra charge",
    "Confirm refund completion with customer",
    "Escalate issue to billing manager",
    "Document dispute threat"
  ],
  "needs_human": true
}

INPUT: Sure! Here you go: {"category": "billing", "priority": "supe 
 -> parsed: False | error: 5 validation errors for Ticket priority   Input should be 'low', 'medium', 'high' or 'urge

INPUT: ```json {"category": "billing"} ``` 
 -> parsed: False | error: 5 validation errors for Ticket priority   Field required [type=missing, input_value={'cate

INPUT: not json at all 
 -> parsed: False | error: 1 validation error for Ticket   Invalid JSON: expected ident at line 1 column 2 [type=json


## 4. Does few-shot really help? Measure it.
Same ticket, 5 runs at temperature 0.8. A consistent prompt should give the same label every time.

In [ ]:
def consistency(ticket_text, use_few_shot, n=5):
    labels, valid = [], 0
    for _ in range(n):
        raw = call_llm(build_messages(ticket_text, use_few_shot), json_mode=True, temperature=0.8)
        p, _ = safe_parse(raw)
        if p:
            valid += 1
            labels.append((p.category, p.priority, p.needs_human))
        time.sleep(0.4)
    return valid, Counter(labels)

for flag in (False, True):
    valid, counts = consistency(SAMPLE, use_few_shot=flag)
    print(f"few_shot={flag} | valid {valid}/5 | label distribution: {dict(counts)}")

few_shot=False | valid 5/5 | label distribution: {('billing', 'urgent', True): 5}
few_shot=True | valid 5/5 | label distribution: {('billing', 'urgent', True): 5}


## 5. Batch evaluation: bad prompt vs improved prompt
Metric: % of outputs that pass schema validation.

In [ ]:
TEST_TICKETS = [
    SAMPLE,
    "App crashes every time I open the camera feature on Android 14. Tried reinstalling twice.",
    "How do I change the email address on my account?",
    "Your service has been completely down for 2 hours and we are losing customers! FIX THIS NOW.",
    "Love the product. Is there an annual plan discount?",
    "I received the wrong item. I ordered a blue jacket and got a red one.",
]

bad_ok = good_ok = 0
for t in TEST_TICKETS:
    bad_raw = call_llm([{"role": "user", "content": f"Analyze this support ticket: {t}"}], temperature=0.7)
    if safe_parse(bad_raw)[0]:
        bad_ok += 1
    good = triage_ticket(t)
    if good["ok"]:
        good_ok += 1
    time.sleep(0.4)

n = len(TEST_TICKETS)
print(f"Bad prompt      : {bad_ok}/{n} valid structured outputs")
print(f"Improved prompt : {good_ok}/{n} valid structured outputs")

Bad prompt      : 0/6 valid structured outputs
Improved prompt : 6/6 valid structured outputs


## 6. Context window management
Chat history grows. We (1) estimate tokens, (2) when over budget, summarize the oldest turns
into one message and keep the most recent turns verbatim, (3) hard-trim as a last resort.

In [ ]:
def estimate_tokens(messages) -> int:
    # rough rule: ~4 characters per token (good enough for budgeting)
    return sum(len(m["content"]) for m in messages) // 4 + 4 * len(messages)

class ContextManager:
    def __init__(self, system_prompt, max_tokens=1500, keep_last=4):
        self.system = {"role": "system", "content": system_prompt}
        self.history = []
        self.max_tokens = max_tokens
        self.keep_last = keep_last
        self.summary = None

    def _messages(self):
        msgs = [self.system]
        if self.summary:
            msgs.append({"role": "system", "content": f"Summary of earlier conversation: {self.summary}"})
        return msgs + self.history

    def _compress(self):
        if len(self.history) <= self.keep_last:
            return
        old, recent = self.history[:-self.keep_last], self.history[-self.keep_last:]
        transcript = "\n".join(f"{m['role']}: {m['content']}" for m in old)
        prev = f"Previous summary: {self.summary}\n" if self.summary else ""
        try:
            self.summary = call_llm([
                {"role": "system", "content": "Summarize this support conversation in under 80 words. Keep ticket IDs, names, amounts, and unresolved issues."},
                {"role": "user", "content": prev + transcript},
            ], max_tokens=200)
            self.history = recent
        except Exception:
            self.history = recent          # fallback: plain trim

    def add(self, role, content):
        self.history.append({"role": role, "content": content})
        if estimate_tokens(self._messages()) > self.max_tokens:
            self._compress()
        # last-resort hard trim: drop oldest turns until under budget
        while estimate_tokens(self._messages()) > self.max_tokens and len(self.history) > 2:
            self.history.pop(0)

    def reply(self, user_text):
        self.add("user", user_text)
        out = call_llm(self._messages(), max_tokens=200, temperature=0.3)
        self.add("assistant", out)
        return out

ctx = ContextManager("You are a concise, friendly support assistant.", max_tokens=700, keep_last=4)
turns = [
    "Hi, my name is Ali and my order #88213 was charged twice.",
    "The amount was $29 each time, on 3rd and 4th October.",
    "I paid with a Visa card ending 4242.",
    "I also want to update my shipping address to Lahore.",
    "Also the invoice PDF shows the wrong company name.",
    "Can you remind me what my order number and the issue were?",
]
for t in turns:
    reply = ctx.reply(t)
    print(f"[tokens≈{estimate_tokens(ctx._messages())}] summary={'yes' if ctx.summary else 'no'} | history msgs={len(ctx.history)}")
print("\nFinal reply:", reply)
print("\nStored summary:", ctx.summary)

[tokens≈167] summary=no | history msgs=2
[tokens≈309] summary=no | history msgs=4
[tokens≈438] summary=no | history msgs=6
[tokens≈570] summary=no | history msgs=8
[tokens≈404] summary=yes | history msgs=4
[tokens≈547] summary=yes | history msgs=6

Final reply: Sure thing! Here’s a quick recap:

**Order number:** **#88213**  
**Current issue:**  

1. **Duplicate charge** – the Visa ending 4242 was billed **$29 on Oct 3** and **$29 on Oct 4**. We’re set to issue a refund (it should appear in 3‑5 business days).  
2. **Shipping address** – you’d like it changed to a new address in **Lahore**.  
3. **Invoice PDF** – the company name on the invoice is incorrect and needs to be updated.

Just let me know the missing details (email address

Stored summary: Ali reported order #88213 was charged twice ($29 on Oct 3 and $29 on Oct 4) using a Visa ending 4242. The agent confirmed the duplicate charges and explained a refund will be issued (3‑5 business days) but is still awaiting Ali’s email add

## Key takeaways
- A schema in the prompt + `json_mode` + Pydantic validation gives output the app can trust.
- Few-shot examples make labels consistent across runs.
- Always parse defensively and retry with the validation error as feedback.
- Keep history under budget: summarize old turns, keep recent turns verbatim.